In [88]:
#import libraries
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np


In [89]:
texts = [    "I love this movie",
    "This film is amazing",
    "Very good acting",
    "Excellent story",
    "I hate this movie",
    "Terrible film",
    "Very boring story",
    "Worst acting ever"
]

In [90]:
#Creating label
labels = np.array([1,1,1,1,0,0,0,0,])

In [91]:
#Token Embedding
#converting the text into tokens then vectorzing
vocab_size = 1000
max_length = 6

tokenizer = Tokenizer(num_words=vocab_size,oov_token="<OOV>"
)
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

X = pad_sequences(sequences,maxlen=max_length,padding= "post")
print(tokenizer.word_index)

print("Input sequence")
print(X)

{'<OOV>': 1, 'this': 2, 'i': 3, 'movie': 4, 'film': 5, 'very': 6, 'acting': 7, 'story': 8, 'love': 9, 'is': 10, 'amazing': 11, 'good': 12, 'excellent': 13, 'hate': 14, 'terrible': 15, 'boring': 16, 'worst': 17, 'ever': 18}
Input sequence
[[ 3  9  2  4  0  0]
 [ 2  5 10 11  0  0]
 [ 6 12  7  0  0  0]
 [13  8  0  0  0  0]
 [ 3 14  2  4  0  0]
 [15  5  0  0  0  0]
 [ 6 16  8  0  0  0]
 [17  7 18  0  0  0]]


In [92]:
#Positional Embedding
#Model tries to learn/know the sequence order of words
class TokenAndPositionEmbedding(layers.Layer): #1
    def __init__(self, max_length, vocab_size, embed_dim):
        super().__init__()
        self.token_embedding = layers.Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim
        )

        self.position_embedding = layers.Embedding(
            input_dim=max_length,
            output_dim=embed_dim
        )

    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[-1], delta=1)
        token_emb = self.token_embedding(x)
        position_emb = self.position_embedding(positions)
        return token_emb + position_emb

In [93]:
#Multi-head Attention
#  The attention allows the model to focus on different parts of the input sequence when making predictions,
#  feed forward network helps in learning patterns and relationships in the data.
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        #multi_head attention
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim)
        ])

        self.layernorm1 = layers.LayerNormalization()
        self.layernorm2 = layers.LayerNormalization()

    def call(self, inputs):
        # Self-attention (Query, Key, Value are calculated here using the same input)
        # The attention layer takes the input and computes
        # attention scores to capture relationships between different positions in the sequence.
        attention_output = self.attention(inputs, inputs)

        # Add + Normalize
        out1 = self.layernorm1(inputs + attention_output)

        # Feed-forward network
        # The output form the previous node is passed to the next node
        ffn_output = self.ffn(out1)

        # Add + Normalize
        # residual connection followed by layer normalization applied after each Transformer sublayer
        out2 = self.layernorm2(out1 + ffn_output)

        return out2

In [101]:
embed_dim = 1000
num_heads = 16
ff_dim = 32
inputs = layers.Input(shape=(max_length,))

x = TokenAndPositionEmbedding(
    max_length=max_length,
    vocab_size=vocab_size,
    embed_dim=embed_dim)(inputs)

x = TransformerBlock(
    embed_dim=embed_dim,
    num_heads=num_heads,
    ff_dim=ff_dim
)(x)

#output layer

x = layers.GlobalAveragePooling1D()(x)


outputs = layers.Dense(1, activation="sigmoid")(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs)

In [102]:
model.compile(optimizer="adam",loss="binary_crossentropy",metrics=['accuracy'])

model.summary()

Model: "functional_23"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_22 (InputLayer)     │ (None, 6)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding_11 │ (None, 6, 1000)        │     1,006,000 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_11            │ (None, 6, 1000)        │    64,118,032 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_11     │ (None, 1000)           │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_35 (Dense)                │ (None, 1)              │         1,001 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 65,125,033 (248.43 MB)

 Trainable params: 65,125,033 (248.43 MB)

 Non-trainable params: 0 (0.00 B)

In [104]:
import time
start = time.time()
model.fit(X,labels,epochs=30,batch_size=2,verbose = 0)
end = time.time()
print(f"{end-start:.2f}")

335.61


In [103]:
test_sentences = [
    "I love the film",
    "This movie was awful"
]

test_seq = tokenizer.texts_to_sequences(test_sentences)
test_pad = pad_sequences(test_seq, maxlen=max_length, padding="post")
predictions = model.predict(test_pad)

for sentence, prediction in zip(test_sentences, predictions):
    print(sentence, "->", prediction[0])
    if prediction[0] > 0.5:
        print("Prediction: Positive")
    else:
        print("Prediction: Negative")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 442ms/step
I love the film -> 0.49665886
Prediction: Negative
This movie was awful -> 0.6101511
Prediction: Positive


Input Text - Text Preprocessing (Tokenization + Padding) - Embedding Layer
(Token Embedding + Position Embedding) - Transformer Encoder (Multi-Head Attention + Feed Forward Network) - Pooling Layer (Global Average Pooling) - Classification Layer (Dense + Sigmoid) - Sentiment Prediction (Positive / Negative)